# Day 7 — Preparing v3 Data (RAG-Aware Dataset Compilation)

Welcome to the **Day 7 Dataset Preparation Notebook** (Jira **KAN-40**)! Today we will:
1. **Mount Google Drive** to access our existing dataset splits and models.
2. Merge the synthetic Q&A data with the original dataset.
3. Generate a structured **`'context'`** passage for each sample to support **Retrieval-Augmented Generation (RAG)** training.
4. Compile **`train_v3.json`**, **`val_v3.json`**, and **`test_v3.json`**.
5. Generate the **v3 LoRA configuration files** for Qwen and Llama.
6. Sync the new datasets and configuration files back to Google Drive.

---  
## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---  
## Step 2: Initialize Directories & Pull Existing Dataset Files

In [ ]:
import os

project_dir = "/content/Retail"
gdrive_dir = None
for candidate in ["/content/drive/MyDrive/Retail LLM", "/content/drive/MyDrive/Retail"]:
    if os.path.isdir(candidate):
        gdrive_dir = candidate
        break
if gdrive_dir is None:
    gdrive_dir = "/content/drive/MyDrive/Retail LLM"
    print(f"[!] Project folder not found on Drive. Defaulting target back to: {gdrive_dir}")
else:
    print(f"[+] Detected active Drive folder: {gdrive_dir}")

for d in ["data/raw", "data/processed", "configs", "src", "models"]:
    os.makedirs(os.path.join(project_dir, d), exist_ok=True)

drive_processed = os.path.join(gdrive_dir, "data", "processed")
local_processed = os.path.join(project_dir, "data", "processed")

if os.path.isdir(drive_processed):
    print("[*] Copying data splits from Google Drive...")
    !cp -v "{drive_processed}/"*.json "{local_processed}/" 2>/dev/null || true
print("[+] Data synchronization complete.")

---  
## Step 3: Write Latest Data Prep, Training & Evaluation Scripts to Workspace

In [ ]:
prepare_code = "import os\nimport json\nimport random\nimport re\n\ndef extract_context_from_sample(instruction, response, item):\n    \"\"\"\n    Synthesizes a declarative, fact-grounded context passage from the instruction and response.\n    Acts as the retrieved reference document passage for RAG-aware training.\n    \"\"\"\n    text_corpus = (instruction + \" \" + response).lower()\n    \n    # Check domain heuristics\n    is_mfg = (\n        item.get(\"domain\") in [\"manufacturing_process_improvement\", \"lean_six_sigma\", \"statistical_process_control\", \"root_cause_analysis\"]\n        or any(k in text_corpus for k in [\"dmaic\", \"six sigma\", \"spc\", \"x-bar\", \"p-chart\", \"defect\", \"assembly line\", \"root cause\", \"5 whys\", \"pareto\", \"calibration\"])\n    )\n    \n    prefix = \"Process Operations Manual: \" if is_mfg else \"Customer Support Policy Guide: \"\n    \n    # Split response into sentences\n    raw_sentences = [s.strip() for s in re.split(r'(?<=[.!?])\\s+', response) if s.strip()]\n    if not raw_sentences:\n        clean_facts = response.strip()\n    else:\n        # Take up to first 2 sentences containing core instructions/facts\n        candidate = \" \".join(raw_sentences[:min(2, len(raw_sentences))])\n        \n        # Remove colloquial greetings/conversational openers\n        clean_facts = re.sub(\n            r'^(i apologize|we apologize|i am very sorry|thank you for contacting|certainly!?|yes,?|no,?|i have checked|looking at|sure!?|hello,?|hi,?|honored to assist!?|i understand|rest assured,?)\\s*',\n            '',\n            candidate,\n            flags=re.IGNORECASE\n        )\n        \n        # Remove conversational closing questions/disclaimers\n        clean_facts = re.sub(\n            r'(please let us know|feel free to contact|how can i help|is there anything else|thank you for your patience|reach out to us).*$',\n            '',\n            clean_facts,\n            flags=re.IGNORECASE\n        )\n        clean_facts = clean_facts.strip()\n        \n        if not clean_facts:\n            clean_facts = raw_sentences[0]\n\n    # Ensure clean capitalisation and period\n    if clean_facts and not clean_facts[0].isupper():\n        clean_facts = clean_facts[0].upper() + clean_facts[1:]\n    if clean_facts and not clean_facts.endswith('.'):\n        clean_facts += '.'\n        \n    return f\"{prefix}{clean_facts}\"\n\ndef process_dataset(input_path, output_path, is_train=False, synthetic_path=None):\n    \"\"\"\n    Loads dataset, merges synthetic data if training, generates 'context' field for all records, and saves output.\n    \"\"\"\n    records = []\n    if os.path.exists(input_path):\n        print(f\"[*] Loading data from: {input_path}\")\n        with open(input_path, \"r\", encoding=\"utf-8\") as f:\n            records = json.load(f)\n        print(f\"[+] Loaded {len(records)} records from {input_path}\")\n    else:\n        print(f\"[!] Warning: File {input_path} not found.\")\n\n    if is_train and synthetic_path and os.path.exists(synthetic_path):\n        print(f\"[*] Loading synthetic QA pairs from: {synthetic_path}\")\n        with open(synthetic_path, \"r\", encoding=\"utf-8\") as f:\n            synthetic_records = json.load(f)\n        print(f\"[+] Merging {len(synthetic_records)} synthetic records into training set...\")\n        records = records + synthetic_records\n\n    # Add 'context' field to each record\n    v3_records = []\n    for item in records:\n        inst = item.get(\"instruction\", \"\")\n        resp = item.get(\"response\", \"\")\n        \n        # If context does not already exist, generate it\n        ctx = item.get(\"context\")\n        if not ctx:\n            ctx = extract_context_from_sample(inst, resp, item)\n            \n        new_item = {\n            \"instruction\": inst,\n            \"context\": ctx,\n            \"response\": resp\n        }\n        \n        # Preserve original metadata if present\n        for key in [\"category\", \"intent\", \"domain\"]:\n            if key in item:\n                new_item[key] = item[key]\n                \n        v3_records.append(new_item)\n\n    if is_train:\n        random.seed(42)\n        random.shuffle(v3_records)\n\n    os.makedirs(os.path.dirname(output_path), exist_ok=True)\n    with open(output_path, \"w\", encoding=\"utf-8\") as f:\n        json.dump(v3_records, f, ensure_ascii=False, indent=2)\n        \n    print(f\"[+] Saved {len(v3_records)} RAG-aware records to: {output_path}\")\n    if v3_records:\n        print(f\"    - Sample Instruction: {v3_records[0]['instruction'][:80]}...\")\n        print(f\"    - Sample Context:     {v3_records[0]['context'][:100]}...\")\n    return len(v3_records)\n\ndef create_v3_configs(project_root):\n    \"\"\"\n    Creates v3 configuration files pointing to models/qwen_v3 and models/llama_v3.\n    \"\"\"\n    print(\"[*] Creating v3 LoRA configuration files...\")\n    configs_dir = os.path.join(project_root, \"configs\")\n    os.makedirs(configs_dir, exist_ok=True)\n    \n    qwen_v3_cfg = {\n        \"model_type\": \"qwen\",\n        \"base_model_name_or_path\": \"Qwen/Qwen2.5-7B-Instruct\",\n        \"peft_config\": {\n            \"r\": 16,\n            \"lora_alpha\": 32,\n            \"lora_dropout\": 0.05,\n            \"bias\": \"none\",\n            \"task_type\": \"CAUSAL_LM\",\n            \"target_modules\": [\n                \"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\", \"gate_proj\", \"up_proj\", \"down_proj\"\n            ]\n        },\n        \"quantization_config\": {\n            \"load_in_4bit\": True,\n            \"bnb_4bit_quant_type\": \"nf4\",\n            \"bnb_4bit_use_double_quant\": True,\n            \"bnb_4bit_compute_dtype\": \"float16\"\n        }\n    }\n    \n    llama_v3_cfg = {\n        \"model_type\": \"llama\",\n        \"base_model_name_or_path\": \"meta-llama/Meta-Llama-3-8B-Instruct\",\n        \"peft_config\": {\n            \"r\": 16,\n            \"lora_alpha\": 32,\n            \"lora_dropout\": 0.05,\n            \"bias\": \"none\",\n            \"task_type\": \"CAUSAL_LM\",\n            \"target_modules\": [\n                \"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\", \"gate_proj\", \"up_proj\", \"down_proj\"\n            ]\n        },\n        \"quantization_config\": {\n            \"load_in_4bit\": True,\n            \"bnb_4bit_quant_type\": \"nf4\",\n            \"bnb_4bit_use_double_quant\": True,\n            \"bnb_4bit_compute_dtype\": \"float16\"\n        }\n    }\n    \n    with open(os.path.join(configs_dir, \"qwen_lora_config_v3.json\"), \"w\", encoding=\"utf-8\") as f:\n        json.dump(qwen_v3_cfg, f, indent=2)\n    with open(os.path.join(configs_dir, \"llama_lora_config_v3.json\"), \"w\", encoding=\"utf-8\") as f:\n        json.dump(llama_v3_cfg, f, indent=2)\n        \n    print(\"[+] configs/qwen_lora_config_v3.json and configs/llama_lora_config_v3.json successfully written.\")\n\ndef main():\n    script_dir = os.path.dirname(os.path.abspath(__file__))\n    project_root = os.path.dirname(script_dir)\n    processed_dir = os.path.join(project_root, \"data\", \"processed\")\n    \n    print(\"\\n=======================================================\")\n    print(\"[*] DAY 7: RAG-Aware Dataset (v3) Compilation\")\n    print(f\"[*] Project Root: {project_root}\")\n    print(\"=======================================================\\n\")\n    \n    # Check for train.json (or fallback to train_v2.json if already merged)\n    train_orig = os.path.join(processed_dir, \"train.json\")\n    synth_path = os.path.join(processed_dir, \"synthetic_qa.json\")\n    \n    # If train.json not available locally, check if train_v2.json exists\n    if not os.path.exists(train_orig) and os.path.exists(os.path.join(processed_dir, \"train_v2.json\")):\n        train_orig = os.path.join(processed_dir, \"train_v2.json\")\n        synth_path = None # already merged in v2\n        \n    # 1. Compile train_v3.json\n    process_dataset(\n        input_path=train_orig,\n        output_path=os.path.join(processed_dir, \"train_v3.json\"),\n        is_train=True,\n        synthetic_path=synth_path\n    )\n    \n    # 2. Compile val_v3.json\n    val_path = os.path.join(processed_dir, \"val.json\")\n    process_dataset(\n        input_path=val_path,\n        output_path=os.path.join(processed_dir, \"val_v3.json\"),\n        is_train=False\n    )\n    \n    # 3. Compile test_v3.json\n    test_path = os.path.join(processed_dir, \"test.json\")\n    process_dataset(\n        input_path=test_path,\n        output_path=os.path.join(processed_dir, \"test_v3.json\"),\n        is_train=False\n    )\n    \n    # 4. Generate configs\n    create_v3_configs(project_root)\n    print(\"\\n[+] Day 7 dataset preparation complete!\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/prepare_v3_data.py", "w", encoding="utf-8") as f:
    f.write(prepare_code)
print("[+] src/prepare_v3_data.py written.")

train_code = "import os\nimport argparse\nimport json\nimport torch\nfrom datasets import load_dataset\nfrom transformers import (\n    AutoModelForCausalLM,\n    AutoTokenizer,\n    BitsAndBytesConfig\n)\nfrom peft import (\n    LoraConfig,\n    get_peft_model,\n    prepare_model_for_kbit_training\n)\nfrom trl import SFTTrainer, SFTConfig\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"QLoRA Fine-Tuning Pipeline for Retail & Manufacturing LLMs\")\n    parser.add_argument(\n        \"--config\",\n        type=str,\n        required=True,\n        help=\"Path to the model configuration JSON file (e.g. configs/qwen_lora_config.json)\"\n    )\n    parser.add_argument(\n        \"--train_file\",\n        type=str,\n        default=\"data/processed/train.json\",\n        help=\"Path to the training data file\"\n    )\n    parser.add_argument(\n        \"--val_file\",\n        type=str,\n        default=\"data/processed/val.json\",\n        help=\"Path to the validation data file\"\n    )\n    parser.add_argument(\n        \"--output_dir\",\n        type=str,\n        default=None,\n        help=\"Directory to save the fine-tuned model checkpoints (defaults to models/{model_type}_v1)\"\n    )\n    parser.add_argument(\n        \"--test_subset\",\n        action=\"store_true\",\n        help=\"If set, runs a quick training check on a tiny dataset slice (50 items) for 5 steps.\"\n    )\n    parser.add_argument(\n        \"--max_train_samples\",\n        type=int,\n        default=None,\n        help=\"If set, limits the training dataset to this number of samples.\"\n    )\n    parser.add_argument(\n        \"--max_val_samples\",\n        type=int,\n        default=None,\n        help=\"If set, limits the validation dataset to this number of samples.\"\n    )\n    return parser.parse_args()\n\ndef format_example(example):\n    \"\"\"\n    Converts a single dataset row into an instruction-tuning prompt string.\n    Supports RAG-aware formatting if 'context' field is present.\n    \"\"\"\n    instruction = example['instruction']\n    response = example['response']\n    context = example.get('context', '').strip()\n    \n    if context:\n        example['text'] = (\n            f\"Below is an instruction that describes a task, paired with an input that provides further context. \"\n            f\"Write a response that appropriately completes the request.\\n\\n\"\n            f\"### Instruction:\\n{instruction}\\n\\n\"\n            f\"### Context:\\n{context}\\n\\n\"\n            f\"### Response:\\n{response}\"\n        )\n    else:\n        example['text'] = (\n            f\"Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n\"\n            f\"### Instruction:\\n{instruction}\\n\\n\"\n            f\"### Response:\\n{response}\"\n        )\n    return example\n\ndef main():\n    args = parse_args()\n    \n    # 1. Load Model Settings Configuration\n    print(f\"[*] Loading configuration from: {args.config}\")\n    if not os.path.exists(args.config):\n        raise FileNotFoundError(f\"[-] Config file not found at {args.config}\")\n        \n    with open(args.config, \"r\", encoding=\"utf-8\") as f:\n        config = json.load(f)\n        \n    model_type = config.get(\"model_type\", \"model\")\n    model_name = config.get(\"base_model_name_or_path\")\n    peft_settings = config.get(\"peft_config\", {})\n    quant_settings = config.get(\"quantization_config\", {})\n    \n    # Auto-assign output directory if not provided\n    if args.output_dir is None:\n        args.output_dir = f\"models/{model_type}_v1\"\n    print(f\"[+] Output Directory: {args.output_dir}\")\n    \n    # 2. Setup Quantization Configuration\n    compute_dtype_str = quant_settings.get(\"bnb_4bit_compute_dtype\", \"bfloat16\")\n    compute_dtype = torch.bfloat16 if compute_dtype_str == \"bfloat16\" else torch.float16\n    \n    if not torch.cuda.is_available():\n        raise RuntimeError(\"[-] CUDA is not available! QLoRA training requires an active GPU runtime.\")\n        \n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=quant_settings.get(\"load_in_4bit\", True),\n        bnb_4bit_quant_type=quant_settings.get(\"bnb_4bit_quant_type\", \"nf4\"),\n        bnb_4bit_use_double_quant=quant_settings.get(\"bnb_4bit_use_double_quant\", True),\n        bnb_4bit_compute_dtype=compute_dtype\n    )\n    \n    # 3. Load Tokenizer & Model\n    print(f\"[*] Loading tokenizer for {model_name}...\")\n    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)\n    tokenizer.padding_side = \"right\" # SFTTrainer requires padding side right\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n        \n    print(f\"[*] Loading base model {model_name} in 4-bit quantization (this will take a few minutes)...\")\n    model = AutoModelForCausalLM.from_pretrained(\n        model_name,\n        quantization_config=bnb_config,\n        device_map=\"auto\",\n        trust_remote_code=True,\n        torch_dtype=torch.float16\n    )\n    print(\"[+] Base model loaded in 4-bit.\")\n    \n    # 1. Force all parameters and buffers in the base model to float16 to prevent bfloat16 propagation\n    for name, param in model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float16)\n    for name, buf in model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float16)\n            \n    # 2. Set model config torch_dtype to float32 so PEFT initializes adapters in float32\n    model.config.torch_dtype = torch.float32\n            \n    # 4. Prepare Model for PEFT/LoRA Training\n    model = prepare_model_for_kbit_training(model)\n    \n    # 5. Configure LoRA\n    print(\"[*] Configuring LoRA Adapter...\")\n    lora_config = LoraConfig(\n        r=peft_settings.get(\"r\", 16),\n        lora_alpha=peft_settings.get(\"lora_alpha\", 32),\n        target_modules=peft_settings.get(\"target_modules\", []),\n        lora_dropout=peft_settings.get(\"lora_dropout\", 0.05),\n        bias=peft_settings.get(\"bias\", \"none\"),\n        task_type=\"CAUSAL_LM\"\n    )\n    # NOTE: We do NOT call get_peft_model() here — SFTTrainer applies it via peft_config\n    print(\"[+] LoRA config ready.\")\n    \n    # 6. Load Dataset\n    print(f\"[*] Loading dataset files: {args.train_file} & {args.val_file}...\")\n    dataset = load_dataset(\n        \"json\",\n        data_files={\n            \"train\": args.train_file,\n            \"validation\": args.val_file\n        }\n    )\n    \n    train_dataset = dataset[\"train\"]\n    val_dataset = dataset[\"validation\"]\n    \n    if args.max_train_samples is not None:\n        train_dataset = train_dataset.select(range(min(len(train_dataset), args.max_train_samples)))\n        print(f\"[+] Sliced train dataset to {len(train_dataset)} samples.\")\n        \n    if args.max_val_samples is not None:\n        val_dataset = val_dataset.select(range(min(len(val_dataset), args.max_val_samples)))\n        print(f\"[+] Sliced val dataset to {len(val_dataset)} samples.\")\n        \n    train_dataset = train_dataset.map(format_example)\n    val_dataset = val_dataset.map(format_example)\n    print(f\"[+] Formatted datasets with 'text' column.\")\n    \n    # 7. Configure Training Arguments\n    if args.test_subset:\n        print(\"\\n==============================================\")\n        print(\"[!] TEST MODE ENABLED: Slicing datasets and steps\")\n        print(\"==============================================\")\n        train_dataset = train_dataset.select(range(min(len(train_dataset), 50)))\n        val_dataset = val_dataset.select(range(min(len(val_dataset), 10)))\n        print(f\"[+] Sliced datasets: Train = {len(train_dataset)} | Val = {len(val_dataset)}\")\n        \n        training_args = SFTConfig(\n            output_dir=args.output_dir,\n            dataset_text_field=\"text\",\n            max_length=512,\n            per_device_train_batch_size=2,\n            per_device_eval_batch_size=2,\n            gradient_accumulation_steps=1,\n            max_steps=5, # Run only 5 steps to verify loop\n            learning_rate=2e-4,\n            logging_steps=1,\n            eval_strategy=\"steps\",\n            eval_steps=1,\n            save_strategy=\"no\",\n            fp16=True,\n            report_to=\"none\", # Disable W&B logging for quick tests\n            remove_unused_columns=False,\n            disable_tqdm=False\n        )\n    else:\n        print(f\"[+] Datasets loaded: Train = {len(train_dataset)} | Val = {len(val_dataset)}\")\n        training_args = SFTConfig(\n            output_dir=args.output_dir,\n            dataset_text_field=\"text\",\n            max_length=512,\n            num_train_epochs=3,\n            per_device_train_batch_size=4,\n            per_device_eval_batch_size=4,\n            gradient_accumulation_steps=4, # Effective batch size = 16\n            learning_rate=2e-4,\n            logging_steps=10,\n            eval_strategy=\"steps\",\n            eval_steps=50,\n            save_strategy=\"steps\",\n            save_steps=100,\n            save_total_limit=1,\n            fp16=True,\n            report_to=\"wandb\" if os.environ.get(\"WANDB_DISABLED\", \"\").lower() != \"true\" else \"none\",\n            lr_scheduler_type=\"cosine\",\n            remove_unused_columns=False\n        )\n        training_args.warmup_ratio = 0.03\n\n    # 8. Initialize SFTTrainer\n    print(\"[*] Initializing SFTTrainer...\")\n    trainer = SFTTrainer(\n        model=model,\n        train_dataset=train_dataset,\n        eval_dataset=val_dataset,\n        peft_config=lora_config,\n        processing_class=tokenizer,\n        args=training_args\n    )\n    \n    # Force cast any remaining bfloat16 parameters or buffers inside the trainer model to float32\n    # to prevent bfloat16 gradient scaling crash on T4 GPU.\n    for name, param in trainer.model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float32)\n    for name, buf in trainer.model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float32)\n            \n    # 9. Launch Training\n    print(\"[*] Starting training...\")\n    \n    resume_from_checkpoint = None\n    if args.output_dir and os.path.exists(args.output_dir):\n        checkpoints = [\n            d for d in os.listdir(args.output_dir)\n            if d.startswith(\"checkpoint-\") and os.path.isdir(os.path.join(args.output_dir, d))\n        ]\n        if checkpoints:\n            checkpoints.sort(key=lambda x: int(x.split(\"-\")[-1]))\n            latest_checkpoint = os.path.join(args.output_dir, checkpoints[-1])\n            print(f\"[+] Found existing checkpoints. Resuming training from: {latest_checkpoint}\")\n            resume_from_checkpoint = latest_checkpoint\n            \n    trainer.train(resume_from_checkpoint=resume_from_checkpoint)\n    print(\"[+] Training completed successfully!\")\n    \n    # Save final adapter weights\n    print(f\"[*] Saving adapter weights to: {args.output_dir}\")\n    trainer.model.save_pretrained(args.output_dir)\n    tokenizer.save_pretrained(args.output_dir)\n    print(\"[+] Saving complete.\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/train.py", "w", encoding="utf-8") as f:
    f.write(train_code)
print("[+] src/train.py (RAG-aware) written.")

evaluate_code = "import os\nimport argparse\nimport json\nimport torch\nimport random\nfrom datasets import load_dataset\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nfrom peft import PeftModel\nfrom rouge_score import rouge_scorer\nimport nltk\nfrom nltk.translate.bleu_score import sentence_bleu, SmoothingFunction\n\n# Download NLTK data if not present (handled quietly)\ntry:\n    nltk.data.find('tokenizers/punkt')\nexcept LookupError:\n    nltk.download('punkt', quiet=True)\ntry:\n    nltk.data.find('tokenizers/punkt_tab')\nexcept LookupError:\n    nltk.download('punkt_tab', quiet=True)\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"Evaluate fine-tuned model against reference dataset.\")\n    parser.add_argument(\n        \"--model_id\",\n        type=str,\n        required=True,\n        help=\"Hugging Face base model identifier (e.g. Qwen/Qwen2.5-7B-Instruct or meta-llama/Meta-Llama-3-8B-Instruct)\"\n    )\n    parser.add_argument(\n        \"--adapter_dir\",\n        type=str,\n        default=None,\n        help=\"Path to the LoRA adapter directory. If None, evaluates the base model only.\"\n    )\n    parser.add_argument(\n        \"--test_file\",\n        type=str,\n        default=\"data/processed/test.json\",\n        help=\"Path to the test JSON file.\"\n    )\n    parser.add_argument(\n        \"--output_file\",\n        type=str,\n        required=True,\n        help=\"Path to save the evaluation results JSON file.\"\n    )\n    parser.add_argument(\n        \"--num_samples\",\n        type=int,\n        default=100,\n        help=\"Number of random samples to evaluate (default: 100).\"\n    )\n    parser.add_argument(\n        \"--seed\",\n        type=int,\n        default=42,\n        help=\"Random seed for reproducibility.\"\n    )\n    return parser.parse_args()\n\ndef main():\n    args = parse_args()\n    random.seed(args.seed)\n    \n    print(\"\\n==============================================\")\n    print(f\"[*] Base Model: {args.model_id}\")\n    print(f\"[*] Adapter Path: {args.adapter_dir}\")\n    print(f\"[*] Output Path: {args.output_file}\")\n    print(\"==============================================\\n\")\n    \n    # 1. Load Tokenizer\n    print(\"[*] Loading tokenizer...\")\n    tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n        \n    # 2. Load Model in 4-bit Quantization (to fit in T4 GPU VRAM)\n    print(\"[*] Loading base model in 4-bit quantization...\")\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_compute_dtype=torch.float16\n    )\n    \n    base_model = AutoModelForCausalLM.from_pretrained(\n        args.model_id,\n        quantization_config=bnb_config,\n        device_map=\"auto\",\n        trust_remote_code=True,\n        torch_dtype=torch.float16\n    )\n    \n    # Force all bfloat16 parameters and buffers in the base model to float16 to prevent bfloat16 propagation\n    for name, param in base_model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float16)\n    for name, buf in base_model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float16)\n            \n    # 3. Load LoRA Adapter if provided\n    if args.adapter_dir:\n        print(f\"[*] Loading LoRA adapter from {args.adapter_dir}...\")\n        model = PeftModel.from_pretrained(base_model, args.adapter_dir)\n    else:\n        print(\"[*] No adapter provided. Evaluating raw base model.\")\n        model = base_model\n        \n    model.eval()\n    \n    # 4. Load Test Dataset\n    print(f\"[*] Loading test file: {args.test_file}...\")\n    if not os.path.exists(args.test_file):\n        raise FileNotFoundError(f\"Test file not found: {args.test_file}\")\n        \n    with open(args.test_file, \"r\", encoding=\"utf-8\") as f:\n        test_data = json.load(f)\n        \n    if len(test_data) > args.num_samples:\n        print(f\"[*] Sampling {args.num_samples} records from {len(test_data)} total test records.\")\n        # Ensure repeatable sampling using seeded random\n        test_samples = random.sample(test_data, args.num_samples)\n    else:\n        print(f\"[*] Using all {len(test_data)} test records.\")\n        test_samples = test_data\n        \n    # 5. Setup Scorers\n    rouge_scorer_inst = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)\n    smoothing = SmoothingFunction().method1\n    \n    results = []\n    total_r1, total_r2, total_rl, total_bleu = 0.0, 0.0, 0.0, 0.0\n    \n    # 6. Evaluation Generation Loop\n    print(\"\\n[*] Starting text generation and evaluation...\")\n    for idx, sample in enumerate(test_samples):\n        instruction = sample[\"instruction\"]\n        reference = sample[\"response\"]\n        context = sample.get(\"context\", \"\").strip()\n        \n        # Build prompt using SFT instruction-tuning prompt template (RAG-aware)\n        if context:\n            prompt = (\n                f\"Below is an instruction that describes a task, paired with an input that provides further context. \"\n                f\"Write a response that appropriately completes the request.\\n\\n\"\n                f\"### Instruction:\\n{instruction}\\n\\n\"\n                f\"### Context:\\n{context}\\n\\n\"\n                f\"### Response:\\n\"\n            )\n        else:\n            prompt = f\"Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n### Instruction:\\n{instruction}\\n\\n### Response:\\n\"\n        \n        inputs = tokenizer(prompt, return_tensors=\"pt\").to(\"cuda\")\n        \n        with torch.no_grad():\n            outputs = model.generate(\n                **inputs,\n                max_new_tokens=150,\n                temperature=0.7,\n                top_p=0.9,\n                do_sample=True,\n                pad_token_id=tokenizer.eos_token_id\n            )\n            \n        # Slice outputs to retrieve only the generated completion (ignoring prompt tokens)\n        prompt_len = inputs.input_ids.shape[1]\n        generation_tokens = outputs[0][prompt_len:]\n        prediction = tokenizer.decode(generation_tokens, skip_special_tokens=True).strip()\n        \n        # Compute ROUGE\n        rouge_scores = rouge_scorer_inst.score(reference, prediction)\n        r1 = rouge_scores['rouge1'].fmeasure\n        r2 = rouge_scores['rouge2'].fmeasure\n        rl = rouge_scores['rougeL'].fmeasure\n        \n        # Compute BLEU (word level)\n        ref_tokens = nltk.word_tokenize(reference.lower())\n        pred_tokens = nltk.word_tokenize(prediction.lower())\n        bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing)\n        \n        # Accumulate scores\n        total_r1 += r1\n        total_r2 += r2\n        total_rl += rl\n        total_bleu += bleu\n        \n        results.append({\n            \"instruction\": instruction,\n            \"reference\": reference,\n            \"prediction\": prediction,\n            \"metrics\": {\n                \"rouge1\": r1,\n                \"rouge2\": r2,\n                \"rougeL\": rl,\n                \"bleu\": bleu,\n                \"length\": len(prediction)\n            }\n        })\n        \n        if (idx + 1) % 10 == 0 or (idx + 1) == len(test_samples):\n            print(f\"    Processed {idx + 1}/{len(test_samples)} samples...\")\n            \n    # Calculate Summary Scores\n    num_evaluated = len(test_samples)\n    summary = {\n        \"mean_rouge1\": total_r1 / num_evaluated,\n        \"mean_rouge2\": total_r2 / num_evaluated,\n        \"mean_rougeL\": total_rl / num_evaluated,\n        \"mean_bleu\": total_bleu / num_evaluated\n    }\n    \n    output_data = {\n        \"model_id\": args.model_id,\n        \"adapter_dir\": args.adapter_dir,\n        \"summary\": summary,\n        \"results\": results\n    }\n    \n    # 7. Write Results\n    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)\n    with open(args.output_file, \"w\", encoding=\"utf-8\") as f:\n        json.dump(output_data, f, ensure_ascii=False, indent=2)\n        \n    print(\"\\n========================= SUMMARY =========================\")\n    print(f\"[+] ROUGE-1 F-Measure: {summary['mean_rouge1']:.4f}\")\n    print(f\"[+] ROUGE-2 F-Measure: {summary['mean_rouge2']:.4f}\")\n    print(f\"[+] ROUGE-L F-Measure: {summary['mean_rougeL']:.4f}\")\n    print(f\"[+] BLEU Score:        {summary['mean_bleu']:.4f}\")\n    print(\"===========================================================\\n\")\n    print(f\"[+] Detailed evaluation records saved to: {args.output_file}\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/evaluate.py", "w", encoding="utf-8") as f:
    f.write(evaluate_code)
print("[+] src/evaluate.py (RAG-aware) written.")

---  
## Step 4: Execute RAG-Aware Dataset Generation (train_v3.json, val_v3.json, test_v3.json)

In [ ]:
!python /content/Retail/src/prepare_v3_data.py

---  
## Step 5: Verify RAG Dataset Quality & Context Fields

In [ ]:
import json
import os

v3_path = "/content/Retail/data/processed/train_v3.json"
if os.path.exists(v3_path):
    with open(v3_path, "r", encoding="utf-8") as f:
        v3_data = json.load(f)
    print(f"[+] Verification Successful!")
    print(f"    - Total train_v3.json items: {len(v3_data)}")
    print(f"    - Has 'context' key: {'context' in v3_data[0]}")
    print("\n--- Sample RAG-Aware Record ---")
    print(f"Instruction:\n{v3_data[0]['instruction']}\n")
    print(f"Context:\n{v3_data[0]['context']}\n")
    print(f"Response:\n{v3_data[0]['response']}\n")
else:
    print("[X] Error: train_v3.json not found.")

---  
## Step 6: Sync train_v3 Datasets and v3 Configs to Google Drive

In [ ]:
drive_processed = os.path.join(gdrive_dir, "data", "processed")
drive_configs = os.path.join(gdrive_dir, "configs")
os.makedirs(drive_processed, exist_ok=True)
os.makedirs(drive_configs, exist_ok=True)

print("[*] Syncing v3 dataset files to Google Drive...")
!cp -v /content/Retail/data/processed/*_v3.json "{drive_processed}/"

print("[*] Syncing v3 configuration files to Google Drive...")
!cp -v /content/Retail/configs/*_v3.json "{drive_configs}/"

print("[+] Backup complete. Google Drive is fully up-to-date for v3 training!")